# 02 - Modeling experiments

**Issues:** #7 (preprocessing pipeline, this section). Baseline, feature engineering, model comparison, class imbalance and tuning (#8 to #13) will be added below as they are done.

**Rules:** only the training and validation sets are used here; the test set stays untouched until the final evaluation (#20). Reusable code lives in `src/`; this notebook imports it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.clean import load_clean_split
from src.features.preprocessing import FEATURES, build_preprocessor, get_xy

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Preprocessing pipeline (issue #7)

In [2]:
train = load_clean_split("train")   # cleaning rule from src/data/clean.py already applied
val = load_clean_split("val")
X_train, y_train = get_xy(train)
X_val, y_val = get_xy(val)
print(f"train: {X_train.shape} | validation: {X_val.shape}")
print(f"features used ({len(FEATURES)}): {FEATURES}")

train: (19443, 11) | validation: (6482, 11)
features used (11): ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_grade', 'cb_person_default_on_file', 'person_home_ownership', 'loan_intent']


### Tree-based variant
Raw numeric scales are kept (trees are scale-invariant). The pipeline is **fitted on the training set only**, then applied to validation.

In [3]:
tree_pipe = build_preprocessor("tree")
Xt_train = tree_pipe.fit_transform(X_train)
Xt_val = tree_pipe.transform(X_val)

print(f"output shape: train {Xt_train.shape} | validation {Xt_val.shape}")
print(f"missing values after transform: train {int(Xt_train.isna().sum().sum())} | validation {int(Xt_val.isna().sum().sum())}")
print(f"same columns in train and validation: {list(Xt_train.columns) == list(Xt_val.columns)}")
print()
print(list(Xt_train.columns))

output shape: train (19443, 20) | validation (6482, 20)
missing values after transform: train 0 | validation 0
same columns in train and validation: True

['person_income', 'person_emp_length', 'person_emp_length_missing', 'person_age', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_grade', 'cb_person_default_on_file', 'person_home_ownership_MORTGAGE', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


### What the pipeline learned (from the training set)

In [4]:
imputer = tree_pipe.named_steps["rate_imputer"]
emp_median = tree_pipe.named_steps["columns"].named_transformers_["emp_length"].named_steps["impute"].statistics_[0]

learned = pd.Series(imputer.group_medians_, name="median loan_int_rate").round(2).to_frame()
display(learned)
print(f"global median loan_int_rate (fallback for unseen grades): {imputer.global_median_:.2f}")
print(f"median person_emp_length: {emp_median:.1f}")

# The learned medians must equal a direct computation on the training set alone.
direct = X_train.groupby("loan_grade")["loan_int_rate"].median()
assert all(abs(direct[g] - m) < 1e-12 for g, m in imputer.group_medians_.items())
print("Check passed: grade medians come from the training set only.")

,median loan_int_rate
A,7.49
B,10.99
C,13.48
D,15.31
E,16.77
F,18.43
G,20.11


global median loan_int_rate (fallback for unseen grades): 10.99
median person_emp_length: 4.0
Check passed: grade medians come from the training set only.


### Missing-value indicator for `person_emp_length`

In [5]:
for name, raw, out in (("train", X_train, Xt_train), ("validation", X_val, Xt_val)):
    print(f"{name:10s} raw missing share = {raw['person_emp_length'].isna().mean():.4f} | indicator share = {out['person_emp_length_missing'].mean():.4f}")

train      raw missing share = 0.0278 | indicator share = 0.0278
validation raw missing share = 0.0287 | indicator share = 0.0287


### Linear variant
`person_income` goes through `log1p`, and all numeric features are standardized. Means and standard deviations are those of the **training** set, so validation values are close to, but not exactly, 0 and 1.

In [6]:
lin_pipe = build_preprocessor("linear")
Xl_train = lin_pipe.fit_transform(X_train)
Xl_val = lin_pipe.transform(X_val)

numeric = ["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
summary = pd.concat(
    {"train": Xl_train[numeric].agg(["mean", "std"]).T, "validation": Xl_val[numeric].agg(["mean", "std"]).T},
    axis=1,
).round(3)
display(summary)

train      validation       
                            mean  std       mean    std
person_age                   0.0  1.0     -0.010  1.005
person_income                0.0  1.0     -0.014  0.994
person_emp_length           -0.0  1.0     -0.014  0.987
loan_amnt                   -0.0  1.0      0.001  0.999
loan_int_rate                0.0  1.0     -0.001  1.006
loan_percent_income         -0.0  1.0      0.015  0.996
cb_person_cred_hist_length  -0.0  1.0      0.002  0.994

### Summary

- The pipeline has **20 output columns** (7 numeric, 1 missing-value indicator, 2 ordinal, 10 one-hot) and leaves no missing value.
- Imputation medians (interest rate by grade, employment length) are learned on the training set and reused unchanged on validation; a direct computation confirms it.
- Two variants share the same code: `tree` (raw scales) and `linear` (log income + standardization).
- Unknown grades raise an error, unknown home ownership or loan intent values are encoded as all zeros (see `tests/test_preprocessing.py`).
- The fitted pipeline can be saved with `save_preprocessor` and reloaded by the API.